# 02 — Per-niche graphs + ligand-receptor scoring

Step 3 of the pipeline. Each niche becomes one PyG `Data` object:

- **Nodes** = cells of the niche. Features = expression (HVG subset by default).
- **Edges** = intra-niche Delaunay edges (cached per patient).
- **Edge features** = euclidean distance + tier-1 LR-consensus score.

The LR score per edge aggregates ligand × receptor expression over all pairs
in the OmniPath consensus database (loaded via LIANA).

In [ ]:
import sys; from pathlib import Path
REPO = Path.cwd().parent.parent
if str(REPO / 'src') not in sys.path: sys.path.insert(0, str(REPO / 'src'))

from ecofoundation.config.schemas import DataConfig, GraphConfig, LRScoringConfig, NicheConfig
from ecofoundation.io.readers import load_anndata
from ecofoundation.niches.assembly import assign_niches
from ecofoundation.graph.construction import build_niche_graphs
import numpy as np

In [ ]:
data_cfg = DataConfig(path=REPO / 'scVI_adata_annotated.h5ad')
adata = load_anndata(data_cfg)
niches, _ = assign_niches(adata, data_cfg, NicheConfig(strategy='knn', knn_k=50))
print('Built', niches.n_niches, 'niches')

## Graph construction

Node features come from `layers['X_exp']`. With `gene_subset='hvg', n_hvg=500`
we use the top-500 highly variable genes — a balance between expressiveness
and tractability for the GNN. Edge features get distance + tier-1 LR score.

In [ ]:
graph_cfg = GraphConfig(
    node_feature_source='expression',
    node_expression_layer='X_exp',
    gene_subset='hvg',
    n_hvg=500,
    edge_topology='delaunay_intra_niche',
    edge_feature_distance=True,
    edge_feature_normalize_distance=True,
    lr_scoring=LRScoringConfig(enabled=True, resource='omnipath_consensus'),
)
result = build_niche_graphs(adata, niches, data_cfg, graph_cfg)
result.summary

## Inspect one graph

Each `Data` object has the usual PyG attributes plus EcoFoundation metadata
(`niche_id`, `patient`, `sample`, `global_cell_indices`).

In [ ]:
g = result.graphs[0]
print('Niche', g.niche_id, '| patient', g.patient, '| sample', g.sample)
print('nodes:', g.num_nodes, '| edges:', g.edge_index.shape[1] // 2)
print('node feature dim:', g.x.shape[1])
print('edge feature dim:', g.edge_attr.shape[1])
print('edge feature names:', result.edge_feature_names)

## Diagnostic plots

In [ ]:
from ecofoundation.reporting.plots import (
    graph_size_distributions, edge_feature_distributions, niche_graph_figure,
)
import torch
fig = graph_size_distributions(result.graphs)
fig

In [ ]:
all_attrs = torch.cat([g.edge_attr for g in result.graphs], dim=0).cpu().numpy()
fig = edge_feature_distributions(all_attrs, result.edge_feature_names)
fig

## Plot one niche-graph

Edge width encodes the chosen edge feature channel (here: LR score).

In [ ]:
coords = np.asarray(adata.obsm['spatial'])[:, :2]
g = result.graphs[0].clone()
g.pos = torch.from_numpy(coords[g.global_cell_indices.cpu().numpy()].astype(np.float32))
fig = niche_graph_figure(g, edge_feature_index=1, edge_feature_name='lr_score_tier1')
fig